In [21]:
import json
import re
import ssl
from collections import Counter
import certifi
from html import unescape
from pathlib import Path
from urllib.parse import urlencode, urljoin
from urllib.request import Request, urlopen


NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
STATIC_MAP_URL = "https://static-maps.yandex.ru/1.x/"
YANDEX_GEOCODER_URL = "https://geocode-maps.yandex.ru/v1/"
YANDEX_GEOCODER_API_KEY = "0aa0ce3c-c817-4d77-9cff-38f4b5a1bb38"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; Lab8Notebook/1.0)"
}
SSL_CONTEXT = ssl.create_default_context(cafile=certifi.where())


def build_url(base_url, params=None):
    if not params:
        return base_url
    return f"{base_url}?{urlencode(params)}"


def fetch_text(url, params=None):
    request = Request(build_url(url, params), headers=HEADERS)
    with urlopen(request, timeout=30, context=SSL_CONTEXT) as response:
        return response.read().decode("utf-8")


def fetch_json(url, params=None):
    return json.loads(fetch_text(url, params))


def download_file(url, filename, params=None):
    request = Request(build_url(url, params), headers=HEADERS)
    with urlopen(request, timeout=30, context=SSL_CONTEXT) as response:
        data = response.read()
    path = Path(filename)
    path.write_bytes(data)
    return path.resolve()


def nominatim_search(query, limit=1):
    data = fetch_json(
        NOMINATIM_URL,
        {
            "q": query,
            "format": "jsonv2",
            "limit": limit,
            "addressdetails": 1,
            "accept-language": "ru",
        },
    )
    if not data:
        raise ValueError(f"Не удалось найти объект по запросу: {query}")
    return data


def record_coords(record):
    return float(record["lat"]), float(record["lon"])

def address_part(record, *keys):
    address = record.get("address", {})
    for key in keys:
        value = address.get(key)
        if value:
            return value
    return None


def federal_district(record):
    return address_part(record, "region")


def region_name(record):
    return address_part(record, "state", "region", "county")


def print_json(record):
    print(json.dumps(record, ensure_ascii=False, indent=2))


def yandex_geocode(query):
    response = fetch_json(
        YANDEX_GEOCODER_URL,
        {
            "apikey": YANDEX_GEOCODER_API_KEY,
            "geocode": query,
            "format": "json",
            "lang": "ru_RU",
        },
    )
    if "response" not in response:
        raise ValueError(response.get("message", "Ошибка Яндекс.Геокодера"))
    return response


def yandex_geoobject(response):
    members = response["response"]["GeoObjectCollection"]["featureMember"]
    if not members:
        raise ValueError("Яндекс.Геокодер не вернул результатов")
    return members[0]["GeoObject"]


def yandex_coords(geoobject):
    lon, lat = map(float, geoobject["Point"]["pos"].split())
    return lat, lon


def yandex_components(geoobject):
    return geoobject["metaDataProperty"]["GeocoderMetaData"]["Address"].get("Components", [])


def yandex_component_name(geoobject, kind):
    for component in yandex_components(geoobject):
        if component.get("kind") == kind:
            return component.get("name")
    return None


def yandex_postcode(geoobject):
    return geoobject["metaDataProperty"]["GeocoderMetaData"]["Address"].get("postal_code")


def yandex_federal_district(geoobject):
    for component in yandex_components(geoobject):
        name = component.get("name", "")
        if "федеральный округ" in name.lower():
            return name
    return None

№1 (2 балла)
Yandex.Maps Static API позволяет получить изображение нужного фрагмента карты, которое можно
разместить на сайте или в приложении. Такое изображение оптимизировано, «весит» не очень
много и загружается быстро даже при медленном Интернете.
Посмотрите, что означают параметры ll, spn, l и c помощью запросов к API через браузер получите:
a) Крупномасштабную схему с КемГУ
b) Крупномасштабную схему района, в котором вы живете
c) Крупномасштабную схему города, в котором вы родились
d) Спутниковый снимок Эйфелевой башни
e) Спутниковый снимок Авачинского вулкана
f) Спутниковый снимок озера Байкал
g) Спутниковый снимок космодрома Байконур


In [22]:
maps_data = {
    "КемГУ": {"ll": "86.0909082,55.3518137", "spn": "0.01,0.006", "l": "map"},
    "Ленинский_район_Кемерово": {"ll": "86.162,55.345", "spn": "0.09,0.05", "l": "map"},
    "Кемерово": {"ll": "86.0872,55.3547", "spn": "0.32,0.18", "l": "map"},
    "Эйфелева_башня": {"ll": "2.2945,48.8584", "spn": "0.01,0.01", "l": "sat"},
    "Авачинский_вулкан": {"ll": "158.8360,53.2570", "spn": "0.08,0.05", "l": "sat"},
    "Озеро_Байкал": {"ll": "108.0,53.5", "spn": "6.0,3.5", "l": "sat"},
    "Космодром_Байконур": {"ll": "63.3070,45.9640", "spn": "0.7,0.45", "l": "sat"}
}

print("\nСсылки для открытия в браузере:\n")

for name, data in maps_data.items():
    url = f"https://static-maps.yandex.ru/1.x/?ll={data['ll']}&spn={data['spn']}&l={data['l']}&size=650,450"
    print(f"{name}:")
    print(f"{url}\n")
    
    response = requests.get(url)
    if response.status_code == 200:
        filename = f"task1_{name}.png"
        with open(filename, "wb") as f:
            f.write(response.content)
        print(f"   также сохранено как {filename}\n")
    else:
        print(f"   ошибка скачивания: {response.status_code}\n")



Ссылки для открытия в браузере:

КемГУ:
https://static-maps.yandex.ru/1.x/?ll=86.0909082,55.3518137&spn=0.01,0.006&l=map&size=650,450

   также сохранено как task1_КемГУ.png

Ленинский_район_Кемерово:
https://static-maps.yandex.ru/1.x/?ll=86.162,55.345&spn=0.09,0.05&l=map&size=650,450

   также сохранено как task1_Ленинский_район_Кемерово.png

Кемерово:
https://static-maps.yandex.ru/1.x/?ll=86.0872,55.3547&spn=0.32,0.18&l=map&size=650,450

   также сохранено как task1_Кемерово.png

Эйфелева_башня:
https://static-maps.yandex.ru/1.x/?ll=2.2945,48.8584&spn=0.01,0.01&l=sat&size=650,450

   также сохранено как task1_Эйфелева_башня.png

Авачинский_вулкан:
https://static-maps.yandex.ru/1.x/?ll=158.8360,53.2570&spn=0.08,0.05&l=sat&size=650,450

   также сохранено как task1_Авачинский_вулкан.png

Озеро_Байкал:
https://static-maps.yandex.ru/1.x/?ll=108.0,53.5&spn=6.0,3.5&l=sat&size=650,450

   также сохранено как task1_Озеро_Байкал.png

Космодром_Байконур:
https://static-maps.yandex.ru/1.x/?ll=

Геокодер помогает определить координаты объекта по его адресу или, наоборот, установить адрес
по координатам. К геокодеру можно также обращаться по протоколу HTTPS.
Для обращения к этому API нужен ключ, бесплатный ключ имеет ряд ограничений
(https://yandex.ru/dev/maps/geocoder/doc/desc/concepts/limits.html), но он полностью подойдет
для наших целей, получить его можно тут https://developer.tech.yandex.ru/.
Ознакомьтесь с документацией, попробуйте сделать запросы, чтобы понять, что означают ответы
геокодера, и ответьте на следующие вопросы (для каждого пункта укажите запрос, который
использовали, и полученный ответ):
a) Получите координаты Якутска и Магадана в формате JSON. Какой город находится севернее:
Якутск или Магадан?
b) Получите координаты вашего родного города и города Торонто в формате JSON. Какой
город из них находится южнее?
c) Определите к каким федеральным округам относятся города: Хабаровск, Уфа, Нижний
Новгород, Калининград, ваш родной город?
d) Узнайте почтовый индекс КемГУ

In [80]:
print("\n2a) Якутск и Магадан:")

yakutsk_resp = yandex_geocode("Якутск")
magadan_resp = yandex_geocode("Магадан")

print("JSON ответ для Якутска:")
print_json(yakutsk_resp)

print("\nJSON ответ для Магадана:")
print_json(magadan_resp)

yakutsk_obj = yandex_geoobject(yakutsk_resp)
magadan_obj = yandex_geoobject(magadan_resp)

yakutsk_pos = yakutsk_obj["Point"]["pos"].split()
magadan_pos = magadan_obj["Point"]["pos"].split()

yakutsk_lon = float(yakutsk_pos[0])
yakutsk_lat = float(yakutsk_pos[1])
magadan_lon = float(magadan_pos[0])
magadan_lat = float(magadan_pos[1])

print(f"\nЯкутск: {yakutsk_lat}, {yakutsk_lon}")
print(f"Магадан: {magadan_lat}, {magadan_lon}")

if yakutsk_lat > magadan_lat:
    print("Вывод: Якутск находится СЕВЕРНЕЕ Магадана")
else:
    print("Вывод: Магадан находится СЕВЕРНЕЕ Якутска")

print("\n2b) Кемерово и Торонто:")

kemerovo_resp = yandex_geocode("Кемерово")
toronto_resp = yandex_geocode("Торонто")

print("JSON ответ для Кемерово:")
print_json(kemerovo_resp)

print("\nJSON ответ для Торонто:")
print_json(toronto_resp)

kemerovo_obj = yandex_geoobject(kemerovo_resp)
toronto_obj = yandex_geoobject(toronto_resp)

kemerovo_pos = kemerovo_obj["Point"]["pos"].split()
toronto_pos = toronto_obj["Point"]["pos"].split()

kemerovo_lon = float(kemerovo_pos[0])
kemerovo_lat = float(kemerovo_pos[1])
toronto_lon = float(toronto_pos[0])
toronto_lat = float(toronto_pos[1])

print(f"\nКемерово: {kemerovo_lat}, {kemerovo_lon}")
print(f"Торонто: {toronto_lat}, {toronto_lon}")

if kemerovo_lat > toronto_lat:
    print("Вывод: Кемерово севернее Торонто")
else:
    print("Вывод: Торонто находится ЮЖНЕЕ Кемерова")

print("\n2c) Федеральные округа:")

cities = ["Хабаровск", "Уфа", "Нижний Новгород", "Калининград", "Кемерово"]
for city in cities:
    resp = yandex_geocode(city)
    obj = yandex_geoobject(resp)
    fd = yandex_federal_district(obj)
    if fd is None:
        fd = yandex_component_name(obj, "province")
    print(f"  {city}: {fd}")

print("\n2d) Почтовый индекс КемГУ:")

kemsu_resp = yandex_geocode("Красная 6, Кемерово")
print("JSON ответ для КемГУ:")
print_json(kemsu_resp)

kemsu_obj = yandex_geoobject(kemsu_resp)
postal = yandex_postcode(kemsu_obj)
print(f"\nПочтовый индекс: {postal}")


2a) Якутск и Магадан:
JSON ответ для Якутска:
{
  "response": {
    "GeoObjectCollection": {
      "metaDataProperty": {
        "GeocoderResponseMetaData": {
          "request": "Якутск",
          "results": "10",
          "found": "10"
        }
      },
      "featureMember": [
        {
          "GeoObject": {
            "metaDataProperty": {
              "GeocoderMetaData": {
                "precision": "other",
                "text": "Россия, Республика Саха (Якутия), Якутск",
                "kind": "locality",
                "Address": {
                  "country_code": "RU",
                  "formatted": "Россия, Республика Саха (Якутия), Якутск",
                  "Components": [
                    {
                      "kind": "country",
                      "name": "Россия"
                    },
                    {
                      "kind": "province",
                      "name": "Дальневосточный федеральный округ"
                    },
           

№3 (1 балл)
Напишите программу, которая на экране распечатает полный адрес и координаты Исторического
музея города Москвы (Красная пл-дь, 1).


In [31]:
resp = yandex_geocode("Красная площадь, 1, Москва")
obj = yandex_geoobject(resp)
address = obj["metaDataProperty"]["GeocoderMetaData"]["text"]
lat, lon = yandex_coords(obj)

print(f"Адрес: {address}")
print(f"Координаты: {lat}, {lon}")

Адрес: Россия, Москва, Красная площадь, 1
Координаты: 55.755277, 37.61768


№4 (1 балл)
Напишите программу, которая распечатает на экране к каким областям относятся города: Барнаул,
Мелеуз, Йошкар-Ола.


In [32]:
def get_region(city):
    resp = yandex_geocode(city)
    obj = yandex_geoobject(resp)
    components = yandex_components(obj)
    for component in components:
        if component.get("kind") == "province":
            return component.get("name")
    return "Не найдено"

for city in ["Барнаул", "Мелеуз", "Йошкар-Ола"]:
    print(f"{city}: {get_region(city)}")

Барнаул: Сибирский федеральный округ
Мелеуз: Приволжский федеральный округ
Йошкар-Ола: Приволжский федеральный округ


№5 (1 балл)
Напишите программу, которая распечатает на экране почтовый индекс Московского Уголовного
Розыска (МУРа) «Петровки, 38».

In [35]:
resp = yandex_geocode("Москва, Петровка, 38")
obj = yandex_geoobject(resp)
postal = yandex_postcode(obj)

print(f"Почтовый индекс Мура: {postal}")

Почтовый индекс Мура: 127006


№6 (1 балл)
Напишите программу, которая загрузит и сохранит в файл спутниковый снимок Австралии
целиком.

In [36]:
params = {
    "ll": "135.0,-25.0",
    "spn": "40.0,30.0",
    "l": "sat",
    "size": "650,450"
}
download_file(STATIC_MAP_URL, "task6_australia.png", params)
print("Снимок Австралии сохранён")


Снимок Австралии сохранён


№7 (1 балл)
Напишите программу, которая загрузит и сохранит в файл карту города Кемерово со следующими
отметками (как ставить отметки посмотрите в документации):
a) ЖД Вокзал
b) Кемеровский кардиологический диспансер
c) Музей-заповедник «Красная Горка»
d) Какой-нибудь парк на ваш выбор

In [42]:
kemerovo = {
    "ЖД вокзал": (86.0746, 55.3337),
    "Кемеровский кардиологический диспансер": (86.1241, 55.3458),
    "Музей-заповедник Красная Горка": (86.0824, 55.3613),
    "Парк Чудес": (86.0768, 55.3544),
}

points = "~".join(
    f"{lon},{lat},pm2rdm"
    for lon, lat in kemerovo.values()
)

map_path = download_file(
    STATIC_MAP_URL,
    "kemerovo.png",
    {
        "ll": "86.0872,55.3547",
        "spn": "0.35,0.22",
        "l": "map",
        "pt": points,
        "size": "650,450",
    },
)

print(map_path)
for title, (lon, lat) in kemerovo.items():
    print(f"{title}: {lat}, {lon}")

C:\Даниил\Учебная деятельность\Учеба\Семестр 4\введение в ис\PythonLaba\kemerovo.png
ЖД вокзал: 55.3337, 86.0746
Кемеровский кардиологический диспансер: 55.3458, 86.1241
Музей-заповедник Красная Горка: 55.3613, 86.0824
Парк Чудес: 55.3544, 86.0768


№8 (1 балл)
Напишите программу, которая загрузит и сохранит в файл карту Кемеровской области целиком, с
нанесенной на нее ломанной линией маршрута: Кемерово – Ленинск-Кузнецк – Новокузнецк –
Шерегеш. Как наносить ломанную линию на карту посмотрите в документации

In [76]:
route_kuzbass = [
    (86.0872, 55.3547),  # Кемерово
    (86.1622, 54.6636),  # Ленинск-Кузнецк
    (87.1099, 53.7576),  # Новокузнецк
    (87.9786, 52.9272),  # Шерегеш
]

pl = "c:ff0000AA,w:5," + ",".join(
    f"{lon},{lat}"
    for lon, lat in route_kuzbass
)

map = download_file(
    STATIC_MAP_URL,
    "task8_kemerovo_route.png",
    {
        "ll": "87.0,54.2",
        "spn": "3.4,3.1",
        "l": "map",
        "pl": pl,
        "size": "650,450",
    },
)
print(map)

C:\Даниил\Учебная деятельность\Учеба\Семестр 4\введение в ис\PythonLaba\task8_kemerovo_route.png


№9 (2 балл)
Напишите программу, которая определяет, какой из списка городов расположен южнее всех
остальных.
Программа должна быть реализована как консольное приложение, список городов вводится с
клавиатуры через запятую.
В результате своей работы программа должна напечатать название самого южного из введённых
городов. 

In [49]:
cities_input = input("Введите города через запятую: ")
cities = [c.strip() for c in cities_input.split(",")]

southernmost = None
min_lat = 90

for city in cities:
    resp = yandex_geocode(city)
    obj = yandex_geoobject(resp)
    pos = obj["Point"]["pos"].split()
    lon = float(pos[0])
    lat = float(pos[1])
    print(f"{city}: {lat}")
    if lat < min_lat:
        min_lat = lat
        southernmost = city

print(f"\nСамый южный город: {southernmost}")

Кемерово: 55.355198
Москва: 55.755864
Таштагол: 52.763918

Самый южный город: Таштагол


№10 (2 балл)
Определите длину пути, заданного последовательностью точек.
Сохраните в файл карту с ломанной линией заданного пути, в его средней точке должна стоять
метка.
Последовательность точек задайте по своему усмотрению, например, список

In [81]:
from math import asin, cos, radians, sin, sqrt

def haversine_km(point_a, point_b):
    lon1, lat1 = point_a
    lon2, lat2 = point_b
    radius = 6371.0
    dlon = radians(lon2 - lon1)
    dlat = radians(lat2 - lat1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    return 2 * radius * asin(sqrt(a))

def path_length_km(points):
    return sum(haversine_km(a, b) for a, b in zip(points, points[1:]))

def middle_point(points):
    total = path_length_km(points)
    half = total / 2
    passed = 0

    for start, end in zip(points, points[1:]):
        segment = haversine_km(start, end)
        if passed + segment >= half:
            ratio = (half - passed) / segment
            lon = start[0] + (end[0] - start[0]) * ratio
            lat = start[1] + (end[1] - start[1]) * ratio
            return lon, lat
        passed += segment

    return points[-1]

cities = ["Кемерово", "Ленинск-Кузнецкий", "Новокузнецк", "Шерегеш"]

path_points = [
    (86.0872, 55.3547),  # Кемерово
    (86.1622, 54.6636),  # Ленинск-Кузнецкий
    (87.1099, 53.7576),  # Новокузнецк
    (87.9786, 52.9272),  # Шерегеш
]

print("Расстояния между городами:")
for i in range(len(path_points) - 1):
    dist = haversine_km(path_points[i], path_points[i + 1])
    print(f"  {cities[i]} - {cities[i + 1]}: {dist:.2f} км")

length = path_length_km(path_points)
print(f"\nОбщая длина пути: {length:.2f} км")

mid_lon, mid_lat = middle_point(path_points)
print(f"Средняя точка маршрута: {mid_lat:.6f}, {mid_lon:.6f}")

pl = "c:00ff00AA,w:5," + ",".join(f"{lon},{lat}" for lon, lat in path_points)
pt = f"{mid_lon},{mid_lat},pm2rdm"

map_path = download_file(
    STATIC_MAP_URL,
    "task10_route.png",
    {
        "ll": "87.0,54.2",
        "spn": "3.4,3.1",
        "l": "map",
        "pl": pl,
        "pt": pt,
        "size": "650,450",
    },
)

print(f"\nКарта сохранена: {map_path}")

Расстояния между городами:
  Кемерово - Ленинск-Кузнецкий: 77.00 км
  Ленинск-Кузнецкий - Новокузнецк: 118.09 км
  Новокузнецк - Шерегеш: 108.86 км

Общая длина пути: 303.95 км
Средняя точка маршрута: 54.088354, 86.763922

Карта сохранена: C:\Даниил\Учебная деятельность\Учеба\Семестр 4\введение в ис\PythonLaba\task10_route.png


№11 (1 балла)
Напишите программу, которая выведет на экран все ссылки со страницы
http://olympus.realpython.org/profiles, ориентируясь на атрибут href у HTML тега a .
Вывод должен быть следующим:
http://olympus.realpython.org/profiles/aphrodite
http://olympus.realpython.org/profiles/poseidon
http://olympus.realpython.org/profiles/dionysus

In [53]:
from html.parser import HTMLParser

class LinkParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.links = []
    
    def handle_starttag(self, tag, attrs):
        if tag == 'a':
            for attr, value in attrs:
                if attr == 'href' and value.startswith('/profiles/'):
                    self.links.append(f"http://olympus.realpython.org{value}")

html = fetch_text("http://olympus.realpython.org/profiles")
parser = LinkParser()
parser.feed(html)

for link in parser.links:
    print(link)

http://olympus.realpython.org/profiles/aphrodite
http://olympus.realpython.org/profiles/poseidon
http://olympus.realpython.org/profiles/dionysus


№12 (2 балла)
Напишите программу, которая вытащит список всех авторов цитат с сайта
https://quotes.toscrape.com/ и выведет его на экран, отсортированным по уменьшению количества
цитат автора, т.е. самым первым должен быть автор с наибольшим числом цитат на сайте. Обратите
внимание, что это многостраничный сайт

In [54]:
from collections import defaultdict
from html.parser import HTMLParser

class AuthorParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.in_author = False
        self.current_author = None
        self.authors = defaultdict(int)
    
    def handle_starttag(self, tag, attrs):
        if tag == 'small' and ('class', 'author') in attrs:
            self.in_author = True
    
    def handle_endtag(self, tag):
        if tag == 'small' and self.in_author:
            self.in_author = False
            if self.current_author:
                self.authors[self.current_author] += 1
                self.current_author = None
    
    def handle_data(self, data):
        if self.in_author:
            self.current_author = data.strip()

author_count = defaultdict(int)

for page in range(1, 100):
    html = fetch_text(f"https://quotes.toscrape.com/page/{page}/")
    parser = AuthorParser()
    parser.feed(html)
    if not parser.authors:
        break
    for author, count in parser.authors.items():
        author_count[author] += count

sorted_authors = sorted(author_count.items(), key=lambda x: x[1], reverse=True)
for author, count in sorted_authors:
    print(f"{author}: {count}")

Albert Einstein: 10
J.K. Rowling: 9
Marilyn Monroe: 7
Dr. Seuss: 6
Mark Twain: 6
Jane Austen: 5
C.S. Lewis: 5
Bob Marley: 3
Eleanor Roosevelt: 2
Ralph Waldo Emerson: 2
Mother Teresa: 2
George R.R. Martin: 2
Ernest Hemingway: 2
Charles Bukowski: 2
Suzanne Collins: 2
André Gide: 1
Thomas A. Edison: 1
Steve Martin: 1
Douglas Adams: 1
Elie Wiesel: 1
Friedrich Nietzsche: 1
Allen Saunders: 1
Pablo Neruda: 1
Garrison Keillor: 1
Jim Henson: 1
Charles M. Schulz: 1
William Nicholson: 1
Jorge Luis Borges: 1
George Eliot: 1
Martin Luther King Jr.: 1
James Baldwin: 1
Haruki Murakami: 1
Alexandre Dumas fils: 1
Stephenie Meyer: 1
Helen Keller: 1
George Bernard Shaw: 1
J.R.R. Tolkien: 1
Alfred Tennyson: 1
Terry Pratchett: 1
J.D. Salinger: 1
George Carlin: 1
John Lennon: 1
W.C. Fields: 1
Ayn Rand: 1
Jimi Hendrix: 1
J.M. Barrie: 1
E.E. Cummings: 1
Khaled Hosseini: 1
Harper Lee: 1
Madeleine L'Engle: 1


№13 (2 балла)
Напишите программу, которая выводит пять случайных цитат с сайта https://quotes.toscrape.com/.
Обратите внимание, что это многостра

In [55]:
import random
from html.parser import HTMLParser

class QuoteParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.in_quote = False
        self.in_author = False
        self.quotes = []
        self.current_quote = None
    
    def handle_starttag(self, tag, attrs):
        if tag == 'span' and ('class', 'text') in attrs:
            self.in_quote = True
        if tag == 'small' and ('class', 'author') in attrs:
            self.in_author = True
    
    def handle_endtag(self, tag):
        if tag == 'span' and self.in_quote:
            self.in_quote = False
        if tag == 'small' and self.in_author:
            self.in_author = False
    
    def handle_data(self, data):
        if self.in_quote:
            self.current_quote = data.strip()
        if self.in_author and self.current_quote:
            self.quotes.append((self.current_quote, data.strip()))
            self.current_quote = None

all_quotes = []

for page in range(1, 100):
    html = fetch_text(f"https://quotes.toscrape.com/page/{page}/")
    parser = QuoteParser()
    parser.feed(html)
    if not parser.quotes:
        break
    all_quotes.extend(parser.quotes)

random_quotes = random.sample(all_quotes, min(5, len(all_quotes)))
for text, author in random_quotes:
    print(f"{text}\n— {author}\n")

“I am good, but not an angel. I do sin, but I am not the devil. I am just a small girl in a big world trying to find someone to love.”
— Marilyn Monroe

“A person's a person, no matter how small.”
— Dr. Seuss

“You may say I'm a dreamer, but I'm not the only one. I hope someday you'll join us. And the world will live as one.”
— John Lennon

“Of course it is happening inside your head, Harry, but why on earth should that mean that it is not real?”
— J.K. Rowling

“Remember, if the time should come when you have to make a choice between what is right and what is easy, remember what happened to a boy who was good, and kind, and brave, because he strayed across the path of Lord Voldemort. Remember Cedric Diggory.”
— J.K. Rowling



№14 (3 балла)
Напишите программу, которая принимает от пользователя теги (их может быть несколько
одновременно). Программа должна вывести все цитаты с этими тегами с сайта
https://quotes.toscrape.com/. Обратите внимание, что это многостраничный сайт

In [59]:
from html.parser import HTMLParser

class TagQuoteParser(HTMLParser):
    def __init__(self, search_tags):
        super().__init__()
        self.search_tags = [t.lower() for t in search_tags]
        self.in_quote = False
        self.in_author = False
        self.in_tags = False
        self.results = []
        self.current_quote = None
        self.current_author = None
        self.current_tags = []
    
    def handle_starttag(self, tag, attrs):
        if tag == 'span' and ('class', 'text') in attrs:
            self.in_quote = True
        if tag == 'small' and ('class', 'author') in attrs:
            self.in_author = True
        if tag == 'div' and ('class', 'tags') in attrs:
            self.in_tags = True
    
    def handle_endtag(self, tag):
        if tag == 'span' and self.in_quote:
            self.in_quote = False
        if tag == 'small' and self.in_author:
            self.in_author = False
        if tag == 'div' and self.in_tags:
            self.in_tags = False
            if any(tag in self.current_tags for tag in self.search_tags):
                self.results.append((self.current_quote, self.current_author, self.current_tags.copy()))
            self.current_quote = None
            self.current_author = None
            self.current_tags = []
    
    def handle_data(self, data):
        if self.in_quote:
            self.current_quote = data.strip()
        if self.in_author:
            self.current_author = data.strip()
        if self.in_tags and not self.in_author and not self.in_quote:
            tag = data.strip()
            if tag:
                self.current_tags.append(tag)

tags_input = input("Введите теги через запятую: ")
search_tags = [t.strip() for t in tags_input.split(",")]

results = []

for page in range(1, 100):
    html = fetch_text(f"https://quotes.toscrape.com/page/{page}/")
    parser = TagQuoteParser(search_tags)
    parser.feed(html)
    if not parser.results and page > 1:
        break
    results.extend(parser.results)

for text, author, tags in results:
    print(f"{text}\n— {author}\nТеги: {', '.join(tags)}\n")

“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”
— Albert Einstein
Теги: Tags:, inspirational, life, live, miracle, miracles

“It is better to be hated for what you are than to be loved for what you are not.”
— André Gide
Теги: Tags:, life, love

“This life is what you make it. No matter what, you're going to mess up sometimes, it's a universal truth. But the good part is you get to decide how you're going to mess it up. Girls will be your friends - they'll act like it anyway. But just remember, some come, some go. The ones that stay with you through everything - they're your true best friends. Don't let go of them. Also remember, sisters make the best friends in the world. As for lovers, well, they'll come and go too. And baby, I hate to say it, most of them - actually pretty much all of them are going to break your heart, but you can't give up because if you give up, you'll never find your soulmate. Yo

№15 (3 балла)
Напишите программу, которой пользователь вводит цену товара. Программа должна выбрать такой
товар с сайта https://scrapingclub.com/exercise/list_basic/, который ближе всего по своей стоимости
к введенной пользователем цене (модуль разности между стоимостью и введенной ценой
минимальный). Обратите внимание, что это многостраничный сайт. Если таких товаров несколько,
то выбирается тот, который по алфавиту находится раньше всех. Программа выводит на экран
следующие данные о найденном товаре: название, описание, фотография, це

In [64]:
import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

response = requests.get("https://scrapingclub.com/exercise/list_basic/", headers=HEADERS)
print(f"Статус: {response.status_code}")
print(f"Длина HTML: {len(response.text)}")
print("Первые 500 символов:")
print(response.text[:500])

Статус: 404
Длина HTML: 1272
Первые 500 символов:
<!DOCTYPE html>


<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Page Not Found 404</title>
    <meta name="description" content="Sorry, the page you were looking for doesn't exist or has been moved. It’s possible the link you followed was broken, or the page may have been removed.">



    <style>
    /* latin */
        body {
            background: #1f1f1f;
            color: white;
            font-siz


Сайт не работает не с впн не  без него